# 🧬 Red Neuronal para Detección de Problemas Cutáneos (VERSIÓN OPTIMIZADA)
# TICS Para La Salud — Universidad del Caribe


In [ ]:

#Instalación y librerías
!pip install kagglehub gradio tensorflow --quiet

import os
import numpy as np
import matplotlib.pyplot as plt
import json
import shutil
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print('✅ Dependencias listas')

✅ Dependencias listas


In [ ]:
# Credenciales Kaggle (Asegúrate de tenerlas en 'Secrets' de Colab)
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')


In [ ]:

# Descarga y Selección de Datos (OPTIMIZADO PARA TIEMPO)
import kagglehub
print('⬇️ Descargando dataset...')
raw_path = kagglehub.dataset_download('ismailpromus/skin-diseases-image-dataset')

# CORRECCIÓN DE RUTA: Entramos a IMG_CLASSES
base_path = os.path.join(raw_path, 'IMG_CLASSES')

# --- OPTIMIZACIÓN DE VELOCIDAD ---
# Creamos un directorio temporal con menos imágenes para que entrene en < 30 min
fast_path = '/content/fast_dataset'
MAX_IMAGES_PER_CLASS = 400 # Reducimos para velocidad extrema

if os.path.exists(fast_path): shutil.rmtree(fast_path)
os.makedirs(fast_path)

print(f'🚀 Seleccionando máximo {MAX_IMAGES_PER_CLASS} imágenes por clase para rapidez...')
for clase in os.listdir(base_path):
    src_dir = os.path.join(base_path, clase)
    dst_dir = os.path.join(fast_path, clase)
    if os.path.isdir(src_dir):
        os.makedirs(dst_dir)
        files = os.listdir(src_dir)[:MAX_IMAGES_PER_CLASS]
        for f in files:
            shutil.copy(os.path.join(src_dir, f), os.path.join(dst_dir, f))

print(f'✅ Dataset optimizado creado en: {fast_path}')

⬇️ Descargando dataset...


100%|██████████| 5.19G/5.19G [01:01<00:00, 90.1MB/s]

Extracting files...


🚀 Seleccionando máximo 400 imágenes por clase para rapidez...
✅ Dataset optimizado creado en: /content/fast_dataset


In [ ]:
# Configuración y Generadores
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    fast_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = train_datagen.flow_from_directory(
    fast_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)

# Definir etiquetas para lógica de riesgo
HIGH_RISK_CLASSES = ['Melanoma', 'Basal Cell Carcinoma', 'Actinic Keratosis']
CLASS_LABELS_ES = {
    'Melanoma': '⚠️ ALTO RIESGO: Posible Melanoma',
    'Basal Cell Carcinoma': '⚠️ ALTO RIESGO: Carcinoma Basocelular',
    'Eczema': 'Eczema / Dermatitis',
    'Atopic Dermatitis': 'Dermatitis Atópica',
    'Psoriasis': 'Psoriasis',
    # Agrega más según las carpetas reales que veas en el print
}


Found 3200 images belonging to 10 classes.
Found 800 images belonging to 10 classes.


In [ ]:

# Modelo (MobileNetV2)
base_model = tf.keras.applications.MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(optimizer=optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Entrenamiento Flash (Fase 1 y 2 combinadas)
print('🚀 Entrenando...')
callbacks = [EarlyStopping(patience=3, restore_best_weights=True)]

# Fase rápida: 5 épocas congelado, 5 épocas fine-tuning
model.fit(train_gen, epochs=5, validation_data=val_gen, callbacks=callbacks)

print('🔓 Descongelando para ajuste fino...')
base_model.trainable = True
for layer in base_model.layers[:-20]: layer.trainable = False
model.compile(optimizer=optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, epochs=5, validation_data=val_gen, callbacks=callbacks)


🚀 Entrenando...
Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 214s 2s/step - accuracy: 0.3925 - loss: 1.8151 - val_accuracy: 0.3650 - val_loss: 1.7265
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.5500 - loss: 1.2529 - val_accuracy: 0.4000 - val_loss: 1.6572
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 211s 2s/step - accuracy: 0.6006 - loss: 1.0694 - val_accuracy: 0.4125 - val_loss: 1.7645
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - accuracy: 0.6400 - loss: 0.9552 - val_accuracy: 0.3975 - val_loss: 1.8136
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 205s 2s/step - accuracy: 0.6888 - loss: 0.8662 - val_accuracy: 0.3963 - val_loss: 1.9210
🔓 Descongelando para ajuste fino...
Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 259s 2s/step - accuracy: 0.4772 - loss: 1.4080 - val_accuracy: 0.3575 - val_loss: 1.9173
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 238s 2s/step - accuracy: 0.5169 - loss: 1.2839 - val_accuracy: 0.3650 - val_loss: 1.9729
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 248s 2s/step 

In [ ]:
# Lógica de Dos Etapas e Interfaz
def predict_skin(image):
    img = Image.fromarray(image).resize(IMG_SIZE)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    probs = model.predict(img_array, verbose=0)[0]
    idx = np.argmax(probs)
    conf = probs[idx]
    label = CLASS_NAMES[idx]

    # Lógica de riesgo
    es_riesgo = any(hr.lower() in label.lower() for hr in HIGH_RISK_CLASSES)

    res = "🔬 RESULTADO:\n"
    if es_riesgo or conf < 0.4:
        res += "⚠️ ATENCIÓN: Se detectaron patrones de posible riesgo o baja certeza.\n"
        res += "Recomendación: Consulte a un dermatólogo a la brevedad."
    else:
        nombre_es = CLASS_LABELS_ES.get(label, label)
        res += f"🩺 Orientación: {nombre_es}\nConfianza: {conf:.1%}"

    return res

import gradio as gr
demo = gr.Interface(fn=predict_skin, inputs=gr.Image(), outputs="text", title="Detección Cutánea - UniCaribe")
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://22f41d3504a829430b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
